In [ ]:
#import
%load_ext autoreload
%autoreload 2

import numpy as np
import src.demo as demo
import src.viz_utils as viz_utils
import src.utils as utils
import importlib
from src.utils import CanonPart, CanonPartMetadata, get_pointcloud_in_cam_frame, transform_cloud_to_base, remove_outliers
from PIL import Image
import torch
import open3d as o3d
import open3d.visualization.gui as gui
import copy as cp
import pickle
import matplotlib

import itertools

In [ ]:

# Load the pointcloud files (for visualization purposes)
save_name = '/home/rthomp12/fewshot/blue_mug_thin_rack_demo/whole_init_scene_pcls.npz'
scene_pcls = np.load(save_name)
scene_pcls = {k: scene_pcls[k] for k in scene_pcls.keys()}

# Load recorded demo transformation
transform_name = '/home/rthomp12/fewshot/blue_mug_thin_rack_demo/ee_transform.npz'
demo_transform = np.load(transform_name)

init_transform = utils.pos_quat_to_transform(demo_transform['init_pos'],
                                             demo_transform['init_quat'])

final_transform = utils.pos_quat_to_transform(demo_transform['final_pos'],
                                              demo_transform['final_quat'])

ee_transform = np.matmul(final_transform, np.linalg.inv(init_transform))

# Load the warp reconstructions
warps = np.load("/home/rthomp12/fewshot/blue_mug_thin_rack_demo/whole_initial_scene_warps.npz", allow_pickle=True)
warps = {k: warps[k] for k in warps.keys()}

child_params = warps['child_params'].item()
parent_params = warps['parent_params'].item()
child_reconstruction = warps['child_reconstructions'].item()

In [ ]:
#Set the object names and canon files that we're using

# Mug v Rack
parent_object = 'rack'
child_object = 'mug'

parent_model_files = {'rack': }
child_model_files = {'mug':}


# # Teapot v Mug

# parent_object = 'mug'
# child_object = 'teapot'

# parent_model_files = {'mug': }
# child_model_files = {'teapot':}


# # Bowl v Mug

# parent_object = 'mug'
# child_object = 'bowl'

# parent_model_files = {'mug': }
# child_model_files = {'teapot': }

parent_part_models = {part: CanonPart.from_pickle(parent_part_model_files[part]) for part in parent_part_names}
child_part_models = {part: CanonPart.from_pickle(child_part_model_files[part]) for part in child_part_names}


In [ ]:
# Verify the reconstruction and demo transformation

child_params = warps['child_params'].item()
reconstructions = {}
transformed_scene_pcls = {}


child_transform = utils.pos_quat_to_transform(child_params.position, child_params.quat)
new_child_transform = np.matmul(ee_transform, child_transform)
child_params.position, child_params.quat = \
    utils.transform_to_pos_quat(new_child_transform)


transformed_scene_pcls = utils.transform_pcd(scene_pcls,
                                             ee_transform,
                                           )
reconstructions[f'reconstructed_{child_object}'] = \
    child_part_models.to_transformed_pcd(child_params)

viz_utils.show_pcds_plotly(transformed_scene_pcls|reconstructions)


In [ ]:
# Find and save interaction points 

nearby_points_delta = 0.035 # Empirically picked

(
    knns,
    deltas,
    target_indices,
) = demo.save_place_nearby_points_v2(
    child_part_names,
    child_part_models,
    child_params,
    parent_part_names,
    parent_part_models,
    parent_params,
    nearby_points_delta,
)
knn_pickle_file = '/home/rthomp12/fewshot/blue_mug_thin_rack_demo/interaction_points.pkl'

interaction_points = {'knns': knns, 'deltas': deltas, "target_indices": target_indices}
pickle.dump(interaction_points, open(knn_pickle_file, 'wb'))

In [ ]:
# Visualize interaction points 

targets_child = {part: {} for part in child_part_names}
targets_parent = {part: {} for part in child_part_names}

anchors = child_part_model.to_pcd(child_params)[
    knns
]
targets_child = np.mean(
    anchors + deltas, axis=1
)
targets_parent = parent_part_models[
    parent_part
].to_pcd(parent_params)[
    target_indices
] 

child_part_targets = {}   
child_targets_viz = {}
parent_targets_viz = {}


child_transform  = utils.pos_quat_to_transform(child_params.position, 
                                              child_params.quat)
parent_transform  = utils.pos_quat_to_transform(parent_params.position, 
                                          parent_params.quat)
child_targets_viz =  child_targets_viz | {
                     f'child_targets': \
                     utils.transform_pcd(targets_child,
                                         child_part_transform),
                     }
parent_targets_viz = parent_targets_viz | {
                     f'parent_targets': \
                     utils.transform_pcd(targets_parent,
                                         parent_part_transform),
                     }

viz_pcls = scene_pcls | child_targets_viz | parent_targets_viz

viz_utils.show_pcds_plotly(viz_pcls)